In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "DGurgurov/xlm-r_slovak_sentiment"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- The pipeline way (commented out) ---
# sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
#
# print("\n--- Sentiment Analysis Example (Pipeline Way) ---")
#
# text_positive_pipeline = "Tento film bol úžasný!"
# prediction_positive_pipeline = sentiment_pipeline(text_positive_pipeline)
# print(f"\nInput: '{text_positive_pipeline}'")
# print(f"Prediction: {prediction_positive_pipeline}")
#
# text_negative_pipeline = "To bolo hrozné, vôbec sa mi to nepáčilo."
# prediction_negative_pipeline = sentiment_pipeline(text_negative_pipeline)
# print(f"\nInput: '{text_negative_pipeline}'")
# print(f"Prediction: {prediction_negative_pipeline}")
#
# text_neutral_pipeline = "Počasie je dnes tak akurát."
# prediction_neutral_pipeline = sentiment_pipeline(text_neutral_pipeline)
# print(f"\nInput: '{text_neutral_pipeline}'")
# print(f"Prediction: {prediction_neutral_pipeline}")


# --- The manual way ---
print("\n--- Manual Sentiment Analysis Example ---")

# Get label mapping from model config
id2label = model.config.id2label

# Rename labels for clarity
id2label_renamed = {0: 'Negative', 1: 'Positive'}
# You can also conditionally map if the model had more labels or different default names:
# if 0 in id2label and id2label[0] == 'LABEL_0':
#     id2label_renamed[0] = 'Negative'
# if 1 in id2label and id2label[1] == 'LABEL_1':
#     id2label_renamed[1] = 'Positive'

def get_manual_sentiment(text, model, tokenizer, device, label_map):
    # 1. Tokenize the input
    # Using the tokenizer directly as a callable is the recommended modern approach
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding='max_length',
        max_length=512
    ).to(device)

    # 2. Perform model inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 3. Apply softmax to get probabilities
    probabilities = torch.softmax(logits, dim=1)

    # 4. Get the predicted label and its score
    max_probability = torch.max(probabilities).item()
    predicted_class_id = torch.argmax(probabilities).item()
    predicted_label = label_map[predicted_class_id]

    return {'label': predicted_label, 'score': max_probability}

# Example 1: Positive sentiment (manual)
text_positive_manual = "Tento film bol úžasný!"
prediction_positive_manual = get_manual_sentiment(text_positive_manual, model, tokenizer, device, id2label_renamed)
print(f"\nInput (manual way): '{text_positive_manual}'")
print(f"Prediction (manual way): {prediction_positive_manual}")

# Example 2: Negative sentiment (manual)
text_negative_manual = "To bolo hrozné, vôbec sa mi to nepáčilo."
prediction_negative_manual = get_manual_sentiment(text_negative_manual, model, tokenizer, device, id2label_renamed)
print(f"\nInput (manual way): '{text_negative_manual}'")
print(f"Prediction (manual way): {prediction_negative_manual}")

# Example 3: Neutral/Mixed sentiment (manual)
text_neutral_manual = "Počasie je dnes tak akurát."
prediction_neutral_manual = get_manual_sentiment(text_neutral_manual, model, tokenizer, device, id2label_renamed)
print(f"\nInput (manual way): '{text_neutral_manual}'")
print(f"Prediction (manual way): {prediction_neutral_manual}")

Loading DGurgurov/xlm-r_slovak_sentiment...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Manual Sentiment Analysis Example ---

Input (manual way): 'Tento film bol úžasný!'
Prediction (manual way): {'label': 'Positive', 'score': 0.9987194538116455}

Input (manual way): 'To bolo hrozné, vôbec sa mi to nepáčilo.'
Prediction (manual way): {'label': 'Negative', 'score': 0.921668291091919}

Input (manual way): 'Počasie je dnes tak akurát.'
Prediction (manual way): {'label': 'Positive', 'score': 0.9962335228919983}
